### Set context and load silver

In [0]:
spark.sql("USE CATALOG nyc_taxi_project")
spark.sql("USE SCHEMA raw")

from pyspark.sql.functions import col

silver = spark.table("nyc_taxi_project.raw.silver_taxi_trips")
print(f"Silver rows available: {silver.count()}")

Silver rows available: 8473894


### Gold Table 1 — Hourly demand by zone (main dashboard driver)

In [0]:
from pyspark.sql.functions import hour, to_date, count, avg, sum as spark_sum, round as spark_round

gold_hourly_demand = (
    silver
    .withColumn("pickup_hour", hour("tpep_pickup_datetime"))
    .withColumn("pickup_date", to_date("tpep_pickup_datetime"))
    .groupBy("pickup_date", "pickup_hour", "PULocationID", "pickup_borough", "pickup_zone")
    .agg(
        count("*").alias("trip_count"),
        spark_round(avg("trip_duration_min"), 2).alias("avg_duration_min"),
        spark_round(avg("fare_amount"), 2).alias("avg_fare"),
        spark_round(spark_sum("fare_amount"), 2).alias("total_revenue"),
        spark_round(avg("tip_pct"), 2).alias("avg_tip_pct"),
        spark_round(avg("avg_speed_mph"), 2).alias("avg_speed_mph")
    )
)

print(f"Gold hourly demand rows: {gold_hourly_demand.count()}")
display(gold_hourly_demand.limit(10))

Gold hourly demand rows: 209980


pickup_date,pickup_hour,PULocationID,pickup_borough,pickup_zone,trip_count,avg_duration_min,avg_fare,total_revenue,avg_tip_pct,avg_speed_mph
2024-01-01,4,50,Manhattan,Clinton West,37,13.01,16.88,624.6,13.89,14.44
2024-01-02,7,13,Manhattan,Battery Park City,10,18.65,26.87,268.7,19.4,16.7
2024-01-02,8,41,Manhattan,Central Harlem,17,10.82,13.29,226.0,10.54,11.08
2024-01-02,8,82,Queens,Elmhurst,1,35.6,30.5,30.5,0.0,12.64
2024-01-02,14,48,Manhattan,Clinton East,70,16.94,21.68,1517.9,17.9,11.14
2024-01-03,4,82,Queens,Elmhurst,1,6.38,7.9,7.9,39.49,10.34
2024-01-03,6,7,Queens,Astoria,2,16.38,18.75,37.5,0.0,14.08
2024-01-03,8,137,Manhattan,Kips Bay,43,9.51,11.19,481.1,23.76,10.73
2024-01-03,9,68,Manhattan,East Chelsea,71,16.52,16.99,1206.6,19.22,8.45
2024-01-03,9,209,Manhattan,Seaport,7,18.22,21.5,150.5,10.66,11.25


### Gold Table 2 — Daily summary (for trend lines / KPIs)

In [0]:
gold_daily_summary = (
    silver
    .withColumn("pickup_date", to_date("tpep_pickup_datetime"))
    .groupBy("pickup_date")
    .agg(
        count("*").alias("total_trips"),
        spark_round(spark_sum("fare_amount"), 2).alias("total_revenue"),
        spark_round(avg("fare_amount"), 2).alias("avg_fare"),
        spark_round(avg("trip_duration_min"), 2).alias("avg_duration_min"),
        spark_round(avg("tip_pct"), 2).alias("avg_tip_pct"),
        spark_round(avg("trip_distance"), 2).alias("avg_distance")
    )
    .orderBy("pickup_date")
)

display(gold_daily_summary)

pickup_date,total_trips,total_revenue,avg_fare,avg_duration_min,avg_tip_pct,avg_distance
2002-12-31,2,16.5,8.25,15.63,15.0,1.02
2008-12-31,1,11.4,11.4,11.33,17.54,1.62
2009-01-01,4,109.3,27.33,32.28,7.69,5.73
2023-12-31,10,144.1,14.41,10.16,20.31,2.6
2024-01-01,67316,1479485.03,21.98,15.07,19.37,4.32
2024-01-02,70425,1504700.59,21.37,16.02,19.48,4.21
2024-01-03,77389,1551324.62,20.05,15.82,19.94,3.95
2024-01-04,96714,1809586.06,18.71,15.32,20.62,3.35
2024-01-05,96307,1742525.69,18.09,14.67,20.81,3.23
2024-01-06,88553,1559981.11,17.62,13.53,25.17,3.19


### Gold Table 3 — Borough-to-borough flow (for a flow/matrix chart)

In [0]:
gold_borough_flow = (
    silver
    .groupBy("pickup_borough", "dropoff_borough")
    .agg(
        count("*").alias("trip_count"),
        spark_round(avg("fare_amount"), 2).alias("avg_fare"),
        spark_round(avg("trip_distance"), 2).alias("avg_distance")
    )
    .orderBy(col("trip_count").desc())
)

display(gold_borough_flow)

pickup_borough,dropoff_borough,trip_count,avg_fare,avg_distance
Manhattan,Manhattan,7146867,13.26,1.86
Queens,Manhattan,456256,56.2,14.04
Manhattan,Queens,224566,47.73,11.21
Manhattan,Brooklyn,163802,32.42,6.98
Queens,Queens,153067,33.19,7.62
Queens,Brooklyn,115778,56.35,14.15
Brooklyn,Brooklyn,31677,20.17,6.79
Manhattan,Bronx,23714,37.93,10.67
Queens,N/A,21871,107.55,23.26
Unknown,Unknown,19084,18.73,3.15


### Gold Table 4 — Top zones ranking (for a bar chart / leaderboard)

In [0]:
gold_top_zones = (
    silver
    .groupBy("PULocationID", "pickup_borough", "pickup_zone")
    .agg(
        count("*").alias("trip_count"),
        spark_round(spark_sum("fare_amount"), 2).alias("total_revenue"),
        spark_round(avg("tip_pct"), 2).alias("avg_tip_pct")
    )
    .orderBy(col("trip_count").desc())
)

display(gold_top_zones.limit(20))

PULocationID,pickup_borough,pickup_zone,trip_count,total_revenue,avg_tip_pct
161,Manhattan,Midtown Center,418132,6613033.4,21.96
237,Manhattan,Upper East Side South,410376,5166354.15,22.89
132,Queens,JFK Airport,404130,2.568214509E7,14.29
236,Manhattan,Upper East Side North,382395,4979298.84,22.82
162,Manhattan,Midtown East,313332,4818316.67,22.14
230,Manhattan,Times Sq/Theatre District,301480,5505814.47,20.87
186,Manhattan,Penn Station/Madison Sq West,300177,4961312.66,21.02
142,Manhattan,Lincoln Square East,289562,3961749.5,22.72
138,Queens,LaGuardia Airport,274646,1.174486049E7,21.45
163,Manhattan,Midtown North,254707,3975254.01,22.02


### Write all Gold tables as Delta

In [0]:
gold_hourly_demand.write.format("delta").mode("overwrite") \
    .saveAsTable("nyc_taxi_project.raw.gold_hourly_demand")

gold_daily_summary.write.format("delta").mode("overwrite") \
    .saveAsTable("nyc_taxi_project.raw.gold_daily_summary")

gold_borough_flow.write.format("delta").mode("overwrite") \
    .saveAsTable("nyc_taxi_project.raw.gold_borough_flow")

gold_top_zones.write.format("delta").mode("overwrite") \
    .saveAsTable("nyc_taxi_project.raw.gold_top_zones")

print("All gold tables written successfully.")

All gold tables written successfully.


### Verify everything

In [0]:
spark.sql("SHOW TABLES IN nyc_taxi_project.raw").show(truncate=False)

+--------+------------------+-----------+
|database|tableName         |isTemporary|
+--------+------------------+-----------+
|raw     |bronze_taxi_trips |false      |
|raw     |dim_taxi_zones    |false      |
|raw     |gold_borough_flow |false      |
|raw     |gold_daily_summary|false      |
|raw     |gold_hourly_demand|false      |
|raw     |gold_top_zones    |false      |
|raw     |silver_taxi_trips |false      |
+--------+------------------+-----------+



In [0]:
for t in ["gold_hourly_demand", "gold_daily_summary", "gold_borough_flow", "gold_top_zones"]:
    n = spark.table(f"nyc_taxi_project.raw.{t}").count()
    print(f"{t}: {n} rows")

gold_hourly_demand: 209980 rows
gold_daily_summary: 96 rows
gold_borough_flow: 62 rows
gold_top_zones: 258 rows


In [0]:

gold_hourly_demand.write.format("delta").mode("overwrite") \
    .saveAsTable("nyc_taxi_project.raw.gold_hourly_demand")

gold_daily_summary.write.format("delta").mode("overwrite") \
    .saveAsTable("nyc_taxi_project.raw.gold_daily_summary")

gold_borough_flow.write.format("delta").mode("overwrite") \
    .saveAsTable("nyc_taxi_project.raw.gold_borough_flow")

gold_top_zones.write.format("delta").mode("overwrite") \
    .saveAsTable("nyc_taxi_project.raw.gold_top_zones")

print("All gold tables written successfully.")

All gold tables written successfully.


In [0]:
# Optimize file layout for faster queries — especially useful as tables grow
spark.sql("OPTIMIZE nyc_taxi_project.raw.silver_taxi_trips ZORDER BY (PULocationID, tpep_pickup_datetime)")
spark.sql("OPTIMIZE nyc_taxi_project.raw.gold_hourly_demand ZORDER BY (pickup_date)")

print("✅ Optimization complete.")

✅ Optimization complete.
